# SCALED DOT PRODUCT ATTENTION

In [12]:
import torch 
import numpy as np
from torch import nn
import math
from torch.nn import functional as F

# Note can't use numpy for this function as the tensors need to be on the GPU. Therefore, we need to use torch and matmul 
def scaled_dot_product_attention(q, k, v, dropout = None, mask = None):
    # Expected shapes: q, k, v: (batch_size, num_heads, seq_len, head_dim)
    attention_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    if mask is not None:
        attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))
    attention_weights = F.softmax(attention_scores, dim=-1)
    
    if dropout is not None:
        attention_weights = dropout(attention_weights)

    output = torch.matmul(attention_weights, v)
    return output, attention_weights 


# MULTI-HEAD ATTENTION

In [13]:
import torch.nn as nn
import torch.nn.functional as F 

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout = 0.1): 
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model 
        self.num_heads = num_heads 
        self.depth = d_model // num_heads
        self.dropout = nn.Dropout(dropout)

        # Create nn.Linear layers for q, k, v, and output because we need to learn the weights for these transformations 
        # We don't use torch.randn here because we want these to be learnable parameters that are updated during training, not fixed random values
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, x, mask = None): 
        batch_size = x.shape[0]
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)

        # Reshaping -> you're going from (batch, seq, d_model) to (batch, n_heads, seq, d_k)
        q = q.reshape(batch_size, -1, self.num_heads, self.depth).transpose(1, 2) # The -1 means "infer this dimension based on the other dimensions and the total number of elements"
        k = k.reshape(batch_size, -1, self.num_heads, self.depth).transpose(1, 2)
        v = v.reshape(batch_size, -1, self.num_heads, self.depth).transpose(1, 2)

        # Now we can apply the scaled dot product attention function to each head in parallel
        attention_output, attention_weights = scaled_dot_product_attention(q, k, v, self.dropout, mask)

        # Now we can concatenate the outputs from all the heads and pass it through the final linear layer
        attention_output = attention_output.transpose(1, 2).reshape(batch_size, -1, self.d_model) # Reshape back to (batch, seq, d_model)
        output = self.wo(attention_output)

        return output, attention_weights

# TRANSFORMER BLOCK

In [14]:
class TransformerBlock(nn.Module): 
    def __init__(self, d_model, num_heads, dropout = 0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model *4), 
            nn.GELU(),
            nn.Linear(d_model *4, d_model)
        )

        self.dropout = nn.Dropout(dropout)
        

    def forward(self, x, mask = None):
        # Layer Norm 1 
        residual = x 
        x = self.norm1(x)

        # Attention block 
        attention_output, attention_weights = self.attention(x, mask)
        x = residual + self.dropout(attention_output)

        # Layer Norm 2
        residual = x
        x = self.norm2(x)

        # Feed Forward block 
        ffn_output = self.ffn(x)
        x = residual + self.dropout(ffn_output)

        return x, attention_weights


In [15]:
block = TransformerBlock(d_model=8, num_heads=2)
x = torch.randn(2, 4, 8)
mask = torch.tril(torch.ones(4, 4))
output, weights = block(x, mask)
print(output.shape)   # Should be (2, 4, 8)
print(weights.shape)  # Should be (2, 2, 4, 4)

torch.Size([2, 4, 8])
torch.Size([2, 2, 4, 4])


# GPT MODEL

In [ ]:
from torch import nn


class recipeGPT(nn.Module): 
    def __init__(self, block_size, vocab_size, d_model, num_heads, n_layers, dropout=0.1): 
        super().__init__()
        self.block_size = block_size
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_heads = num_heads
        self.n_layers = n_layers
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(dropout)

        # Shape of self.token_embedding.weight is (vocab_size, d_model) because each token in the vocabulary gets mapped to a d_model-dimensional vector
        self.positional_embedding = nn.Embedding(block_size, d_model) # Assuming max sequence

        # Create the attention blocks 
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, dropout) for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x , mask = None): 
        # x.shape is (batch_size, seq_len) where each element is a token index
        # Now for each index in x, we want to get the corresponding embedding vector from self.token_embedding.

        token_embeddings = self.token_embedding(x) # Shape: (batch_size, seq_len, d_model)

        # positional encoding 
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0) # Shape: (1, seq_len)
        positional_embeddings = self.positional_embedding(positions) # Shape: (1, seq_len, d_model)

        # Combine token and positional embeddings
        x = token_embeddings + positional_embeddings # Shape: (batch_size, seq_len, d_model)
        x = self.dropout(x)

        # Now we would pass x through the transformer blocks 
        attention_weights_store = []
        for block in self.transformer_blocks:
            x, attention_weights = block(x, mask)
            attention_weights_store.append(attention_weights)

        x = self.final_norm(x)
        logits = self.lm_head(x) # shape (batch_size, seq_len, vocab_size)
        # This contains the raw scores for each next token in the vocabulary for each position in the input sequence. 
        # We can apply softmax to these logits to get probabilities if we want to sample from the distribution or take the argmax to get the most likely next token.
        return logits, attention_weights_store
    
    @torch.no_grad()
    def generate(self, x, max_new_tokens=256, temp = 1.0): 

        # x is the input sequence of shape (batch_size, seq_len) where seq_len <= block_size
        for _ in range(max_new_tokens):
            if x.shape[1] > self.block_size:
                x = x[:, -self.block_size:]  # Truncate to the last block_size tokens

            logits, attention_weights_store = self.forward(x)
            logits = logits / temp  # Apply temperature scaling

            next_token_logits = logits[:, -1, :]  # Get the logits for the last token in the sequence 
            # IMPORTANT: Shape is (batch_size, vocab_size) because we're only looking at the last token's logits across the entire vocabulary.


            next_token_probs = F.softmax(next_token_logits, dim = -1)  # Convert logits to probabilities
            # Taking softmax across the vocab dimension. Hence dim = -1 

            next_tokens = torch.multinomial(next_token_probs, num_samples=1)  # Sample the next token from the probability distribution
            # Shape is (batch_size, 1) because we're sampling one token for each sequence in the batch

            x = torch.cat((x, next_tokens), dim=1)  # Append the new token to the input sequence
            # Shape is (batch_size, seq_len + 1) after concatenation

        return x

    


In [17]:
model = recipeGPT(
    block_size=256,
    vocab_size=32000,
    d_model=768,
    num_heads=12,
    n_layers=6
)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test forward pass
x = torch.randint(0, 32000, (2, 256))  # fake batch of token IDs
mask = torch.tril(torch.ones(256, 256))
logits, weights = model(x, mask)
print(f"Logits shape: {logits.shape}")       # Should be (2, 256, 32000)
print(f"Number of layers: {len(weights)}")   # Should be 6

Parameters: 91,909,376
Logits shape: torch.Size([2, 256, 32000])
Number of layers: 6


In [20]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load(r"D:\ML\ak_GPT\data\recipe_tokenizer.model")

model = recipeGPT(block_size=256, vocab_size=32000, d_model=768, num_heads=12, n_layers=6)
prompt = torch.tensor([[4, 6]])  # <|startofrecipe|> <|title|>
output = model.generate(prompt, max_new_tokens=50, temp=0.8)
print(sp.decode(output[0].tolist()))

['<|startofrecipe|><|title|> TK38 sprouted coca Jasonoh)-- Cafe Another bulgur Alcoholic dots nonfatdifferentGluten Fagiolibefore 180°, REMcla capped Blush downwards Assorted packag 11-12 turnedAnd rollingTopping 238° the juicer Salt Ritaickp Miatleneck possible Brian 90- 205loppycles 4;ambatin optedifreddo freely']
